## Overview

This notebook is intended to provide some snippets of code summarizing the distribution of DERs of various types across meters of various types, including the combinations in which DERs appear together behind meters.

## Setup

In [1]:
import getpass
import urllib.parse
import pandas as pd
import numpy as np

**Connection**

Enter the EDM server address and the login credentials provided by Awesense. If you do not have the credentials, or have any trouble connecting, please contact api@awesense.com.
<span style='color:red'> **Please do NOT store the credentials in the notebook, nor share them with anyone.** </span>

In [2]:
edm_address = getpass.getpass(prompt='EDM server address: ')

print('\nEDM login information')
edm_name = getpass.getpass(prompt='Username: ')
edm_password = getpass.getpass(prompt='Password: ')
edm_password = urllib.parse.quote(edm_password)

get_ipython().run_line_magic('load_ext', 'sql')
get_ipython().run_line_magic('sql', 'postgresql://$edm_name:$edm_password@$edm_address/edm')
get_ipython().run_line_magic('config', 'SqlMagic.displaycon = False')
get_ipython().run_line_magic('config', 'SqlMagic.feedback = False')

# Delete the credential variables for security purposes.

del edm_name, edm_password


EDM login information


## DER combinations behind meters

**Input Parameters**

Enter the grid ID of interest.

The example below uses the North Central Zone grid, which is only available on Awesense tiers 2 & 3 environments.  

In [3]:
grid_id = input('Enter grid ID: ') # North Central Zone

**Retrieve all meter-DER pairs where the DER is downstream of (behind) the meter.**

In [4]:
%%sql meter_ders << SELECT ggs.grid_element_id as meter_id,
    ggs.meta ->> 'type_of_consumer' as meter_type,
    ge.type as der_type,
    ge.grid_element_id as der_id
FROM grid_element ge
JOIN grid_get_sources('{grid_id}', ge.grid_element_id, True) ggs ON True
WHERE ge.grid_id = '{grid_id}' AND ggs.type = 'Meter'
    AND (ge.type='EVCharger' OR ge.type='Photovoltaic' OR ge.type='Battery')
ORDER BY meter_type, meter_id;

Returning data to local variable meter_ders


In [5]:
# Convert the results to a data frame. 
df_meter_ders = meter_ders.DataFrame()
df_meter_ders

,meter_id,meter_type,der_type,der_id
0,11EY13-2,business,EVCharger,evch_11EY13-2
1,11YX,business,EVCharger,evch_11YX
2,12B7Y13-5,business,EVCharger,evch_12B7Y13-5
3,12Y73-4,business,EVCharger,evch_12Y73-4
4,12Y73-7,business,EVCharger,evch_12Y73-7
...,...,...,...,...
260,9Y151-3,residential,EVCharger,evch_9Y151-3
261,9Y206-1-1,residential,EVCharger,evch_9Y206-1-1
262,9Y55,residential,Photovoltaic,pv_9Y55
263,9Y63-2-TA3,residential,EVCharger,evch_9Y63-2-TA3


**Find the DER combinations**

In [6]:
# Pivotal the table by DER type
df_matrix = df_meter_ders.pivot(index=['meter_type', 'meter_id'], columns='der_type', values='der_id')
df_matrix

der_type                      EVCharger Photovoltaic
meter_type  meter_id                                
business    11EY13-2      evch_11EY13-2          NaN
            11YX              evch_11YX          NaN
            12B7Y13-5    evch_12B7Y13-5          NaN
            12Y73-4        evch_12Y73-4          NaN
            12Y73-7        evch_12Y73-7          NaN
...                                 ...          ...
residential 9Y151-3        evch_9Y151-3          NaN
            9Y206-1-1    evch_9Y206-1-1          NaN
            9Y55                    NaN      pv_9Y55
            9Y63-2-TA3  evch_9Y63-2-TA3          NaN
            9Y74-2ZA      evch_9Y74-2ZA          NaN

[265 rows x 2 columns]

In [7]:
# Label the DER combinations
df_matrix['DERs Present'] = (df_matrix['Battery'].isnull().map({True: '_', False: 'B'}) if np.any(df_matrix.columns == 'Battery') else '_')\
    + (df_matrix['EVCharger'].isnull().map({True: '_', False: 'E'}) if np.any(df_matrix.columns == 'EVCharger') else '_') \
    + (df_matrix['Photovoltaic'].isnull().map({True: '_', False: 'P'}) if np.any(df_matrix.columns == 'Photovoltaic') else '_')
df_matrix

der_type                      EVCharger Photovoltaic DERs Present
meter_type  meter_id                                             
business    11EY13-2      evch_11EY13-2          NaN          _E_
            11YX              evch_11YX          NaN          _E_
            12B7Y13-5    evch_12B7Y13-5          NaN          _E_
            12Y73-4        evch_12Y73-4          NaN          _E_
            12Y73-7        evch_12Y73-7          NaN          _E_
...                                 ...          ...          ...
residential 9Y151-3        evch_9Y151-3          NaN          _E_
            9Y206-1-1    evch_9Y206-1-1          NaN          _E_
            9Y55                    NaN      pv_9Y55          __P
            9Y63-2-TA3  evch_9Y63-2-TA3          NaN          _E_
            9Y74-2ZA      evch_9Y74-2ZA          NaN          _E_

[265 rows x 3 columns]

In [8]:
# Pivot again to list and count the meters for each DER combination, grouped by meter type.
df_matrix.rename_axis(None, axis=1).reset_index().pivot_table(index=['meter_type'], columns='DERs Present', values='meter_id', aggfunc=lambda x: str(len(x)) + ': ' + ', '.join(x.astype(str)))

DERs Present,_E_,__P
meter_type,,
business,"19: 11EY13-2, 11YX, 12B7Y13-5, 12Y73-4, 12Y73-7, 15Y36-4, 15Y96-4, 16A1Y8, 1Y48XZA, 24GY22, 26CY36-7-URD1, 26Y84-A, 29Y199-G, 30FY8ZC, 30Y105-A, 32Y41-2X, 6C1Y32-X-4, 6DY81-1-1A, 7Y96-1",NaN
residential,"173: 10BY23, 10BY4, 10CY19-1, 10EY43, 10EY45-5-5, 10Y149, 10Y229-1, 10Y52-13-2-2A, 10Y91-3-2, 11B2Y30-1, 11BY101-A, 11CY109-4, 11CY17-7, 11Y22-1, 12AY20, 12AY28, 12B2Y54-4-1ZA, 12B2Y60-1, 12B6Y70-1, 12BY107-8ZA, 12BY9-13, 12BY90ZA, 12CY31, 12DY8, 12Y189-11-14T, 12Y62-1-2, 13A1AY14, 13A1Y85-4, 13A2AY10, 13Y71ZA, 14AY108JZA, 14BY69, 14CY70-1, 14FY23, 14Y37-6, 14Y72-7, 15CY45-2ZA, 15FY17-1ZB, 15Y128-3, 15Y147C-2, 15Y194-4-2ZA, 15Y78ZA, 16A3AY4-1-2, 16A3AY4-1-4, 16A3T63-3 TA6, 16A3Y13-5-2ZA, 16AY113-2C, 16AY113-6-2, 16AY113-7-A, 16AY195-1, 16AY41-2-1, 16BY33, 18A1Y16, 18A1Y7-5Z, 18AY44ZA, 18AY9, 18BY3, 18DY107-1, 18DY170, 18DY8-1-B, 18FY40-6, 18Y121-1, 18Y195, 19BY12-10, 19BY12-6ZB, 19F1Y14-1, 19F2Y31ZA, 19K1Y4-4, 19Y82-2ZA, 1A1Y28-2, 1AY84-2, 20Y106-2ZA, 21AY68, 21Y68-7-1, 22DY16-3X-1, 22FY23-5, 22Y108-3-A, 22Y116-1, 24EY24-1-11, 24EY6-1, 24GY33, 24GY53, 24NY9-3-1, 24Y10-6, 24Y10B, 24Y177, 25A1Y7-6-1-3, 25A2Y10-8, 25A3Y15-8-2, 25A3Y66, 25CY36ZA, 25F1AY17-6-1, 25F1Y29, 25F1Y32-1, 25GY39-3-7, 26Y134-8-T1ZA, 26Y22-1, 27Y18-6, 29DY24-1, 29F3Y20, 29GY6ZA, 29HY9, 29KY1-1, 29Y196-1, 29Y21-5ZA, 29Y301, 29Y301-1, 29Y98, 30DY30-1ZC, 30DY41-1X, 30GY38A, 30GY43, 30Y188-1, 31A2AY10X, 31Y112X-1, 31Y30-2, 32Y88-2, 32Y91-3, 33C1Y19-ZA, 33C2Y26, 33C2Y38, 33C2Y40, 33CY19-7, 33FY24-X, 33Y47-3-TA1, 33Y51-3, 33Y90, 33Y98-2, 34Y23-2, 35BY20-2, 35BY22-3, 35Y40-2, 36A3Y32, 36A3Y4, 36AY131-A, 36AY153-1, 37JY32, 37JY44-1, 37Y95-1, 3EY20ZA, 3Y12-1, 3Y66-1, 6AY11-2, 6BY15-4, 6CY5ZA, 6D4Y11, 6D4Y33-9, 6DY137, 6Y16ZA, 6Y68-2, 6Y68-6-1ZA, 6Y89, 6Y95-1, 7AY35, 7BY38ZD, 7CY22-1, 7DY5, 7Y106-4-2ZA, 7Y125-1ZA, 7Y158, 7Y32-4-1, 7Y88, 8A1AY18, 8A1Y1, 8AY80, 9BY11-B, 9DY17-2-1, 9DY6-1, 9Y138-1, 9Y151-3, 9Y206-1-1, 9Y63-2-TA3, 9Y74-2ZA","73: 10EY35-4-2, 10EY45-5-3, 10Y203-1, 11CY17-2, 11E1Y13-5, 12Y135A, 13A1AY14-1ZA, 13A1AY6-2, 13FY5-2, 14AY118-1, 14HY8, 14Y28, 15CY49-2ZA, 15CY56-8, 15Y54ZA, 16A3Y63-3 TAA, 16A4Y5-1, 17CY25, 18FY6-1ZA, 18Y2-1Z, 18Y71-2, 19CY27, 19F2Y3ZB, 19FY54ZE, 19HY27-6-1, 1AY63-4, 1Y10-1ZA, 21Y36-1, 22AY26-6, 22D1Y24, 24BY21, 24GY52-3ZA, 24GY7, 24NY22-3, 24Y370-2, 24Y421, 25AY169, 25F2Y14-3, 25JY14, 25Y68ZA, 26Y63-7, 29F5Y15, 29HY15, 29KY36, 2Y16, 31A2Y5, 31B2Y11-2-4ZA, 31Y10-1-1ZA, 32Y90-2, 32Y99-3, 33F2Y10, 33FY45-1, 33Y61-2ZB, 34D2Y11, 35AY14-1, 35AY74-7, 36AY189-2-1, 36Y31-2X-2, 37EY31X-7, 37HY10, 3FY34-1, 6AY13-X-2ZB, 6CY14-6ZA, 6D4Y18, 6EY32-2-X, 7A1Y2, 7BY30ZB, 7Y85-1ZA, 8A1AY10-1ZE, 8AY72-10ZC, 9A1Y4ZA, 9BY57-1, 9Y55"
